DEPRECATED for birdclef.mel2vec.student_teacher

In [2]:
import multiprocessing as mp
from pathlib import Path

import numpy as np
import polars as pl
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors
from torch.utils.data import DataLoader, Dataset

In [3]:
class STGTEmbeddingDataset(Dataset):
    def __init__(
        self,
        root: Path,
        wordvector_path: Path,
        split: str = "train",
        mask_prob: float = 0.1,
    ):
        self.split = split
        self.mask_prob = mask_prob

        df = pl.read_parquet(root)
        self.wordvectors = KeyedVectors.load(str(wordvector_path))
        self.embedding_dim = self.wordvectors.vector_size

        if split == "train":
            self.df = df.filter(pl.col("part") < 80)
        elif split == "val":
            self.df = df.filter(pl.col("part") >= 80)

    def __len__(self):
        return self.df.select(pl.len()).item()

    def __getitem__(self, idx):
        # Polars row() is slow; direct access is faster for large datasets
        row = self.df[idx]
        tokens = row.get_column("tokens").to_numpy()[0]
        target_logits = row.get_column("target").to_numpy()[0]

        # Apply masking during training
        if self.split == "train" and self.mask_prob > 0:
            num_tokens_to_keep = int(len(tokens) * (1 - self.mask_prob))
            tokens = np.random.choice(tokens, size=num_tokens_to_keep, replace=False)

        # The model will handle the embedding lookup
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(
            target_logits, dtype=torch.float32
        )


class LitStudentModel(L.LightningModule):
    def __init__(
        self,
        embedding_dim: int,
        output_dim: int,
        wordvectors: KeyedVectors,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["wordvectors"])

        # Embedding layer, initialized with your pretrained vectors
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(wordvectors.vectors), freeze=True
        )

        # 1D CNN to process the sequence of embeddings
        self.aggregator = nn.Sequential(
            nn.Conv1d(embedding_dim, embedding_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
            # and then a hidden layer for transfer learning
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
        )

        # Final classification head
        self.classifier = nn.Linear(512, output_dim)

        self.distillation_loss = nn.KLDivLoss(reduction="batchmean")
        self.temperature = 3.0

    def forward(self, tokens):
        # tokens -> embeddings -> cnn -> logits
        embeddings = self.embedding(tokens)  # (batch, seq_len, emb_dim)
        embeddings = embeddings.permute(0, 2, 1)  # (batch, emb_dim, seq_len)
        features = self.aggregator(embeddings).squeeze(-1)  # (batch, 128)
        logits = self.classifier(features)
        return logits

    def _step(self, batch):
        student_tokens, teacher_logits = batch
        student_logits = self.forward(student_tokens)

        # KL Divergence for soft label distillation
        student_log_probs = F.log_softmax(student_logits / self.temperature, dim=1)
        teacher_probs = F.softmax(teacher_logits / self.temperature, dim=1)

        loss = (self.temperature**2) * self.distillation_loss(
            student_log_probs, teacher_probs
        )
        return loss

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
scratch = Path("~/scratch/birdclef/2025").expanduser()
root = scratch / "mel2vec-v1"

wordvector_path = list((root / "word2vec").glob("**/epochs=100/word2vec.wordvectors"))[
    0
].as_posix()
data_path = scratch / "soundscape-token-perch"
batch_size = 64
num_workers = min(8, mp.cpu_count())
logit_dim = 10932

# Instantiate datasets and dataloaders
train_ds = STGTEmbeddingDataset(data_path, wordvector_path, split="train")
val_ds = STGTEmbeddingDataset(data_path, wordvector_path, split="val")

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers
)
val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers)

# Instantiate the model
# Note: You need to know the vocab size and output dim (e.g., from Perch)
model = LitStudentModel(
    embedding_dim=train_ds.embedding_dim,
    output_dim=logit_dim,
    wordvectors=train_ds.wordvectors,
)

# get model information
print(model)

# Instantiate the trainer and start training
trainer = L.Trainer(max_epochs=3, accelerator="auto")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

/storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-2025/.venv/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-202 ...
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


LitStudentModel(
  (embedding): Embedding(16383, 384)
  (aggregator): Sequential(
    (0): Conv1d(384, 384, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): AdaptiveMaxPool1d(output_size=1)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (classifier): Linear(in_features=512, out_features=10932, bias=True)
  (distillation_loss): KLDivLoss()
)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
2025-06-17 03:50:07.709896: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 03:50:20.622632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-17 03:50:21.254681: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-17 03:50:23.251600: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBL

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

ColumnNotFoundError: Caught ColumnNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-2025/.venv/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-2025/.venv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-2025/.venv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/tmp/ipykernel_3873685/2500463827.py", line 28, in __getitem__
    target_logits = row.get_column("target").to_numpy()[0]
  File "/storage/home/hcoda1/8/amiyaguchi3/clef/birdclef-2025/.venv/lib/python3.10/s

Going to write a script instead now.